In [11]:
from sentence_transformers import SentenceTransformer
import pandas as pd

In [3]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8692.77it/s]


In [12]:
df = pd.read_json("./ingest/chunked_data.jsonl", orient="records", lines=True)
df

,uuid,title,doc_id,chunk_index_in_document,text
0,2322a7ba-1b14-4bbb-8080-5bb7776667af,Man About Town,https://en.wikipedia.org/wiki/Man_About_Town_(...,0,"Top Hollywood talent agent, Jack Giamoro (Ben ..."
1,16b41447-6996-436f-9606-91ab3aa8b850,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,0,"Tom Dobbs is host of a satirical news program,..."
2,e2fd6cbe-e717-4c69-a6a4-6a989f5aff19,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,1,and given an injection. The next day she displ...
3,1a203d4d-ac1f-44ce-a23f-0c4340209621,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,2,feeling that is soon justified when she sees D...
4,446f8747-0a69-41b6-9d6c-3540a759bd83,The Marine,https://en.wikipedia.org/wiki/The_Marine,0,"In Iraq, John Triton (John Cena) of the United..."
...,...,...,...,...,...
21887,956f43a5-4044-470a-aa7b-986bc21ea535,Hammett,https://en.wikipedia.org/wiki/Hammett_(film),0,"San Francisco-based Dashiell Hammett, trying t..."
21888,6f6660ae-d54b-4ee4-a4bc-f0387f502326,Hanky Panky,https://en.wikipedia.org/wiki/Hanky_Panky_(198...,0,An architect named Michael Jordon accidentally...
21889,e1e657ca-3f03-4b14-96ce-ce9781767797,Harry Tracy,"https://en.wikipedia.org/wiki/Harry_Tracy,_Des...",0,"By the end of the 19th century, Butch Cassidy,..."
21890,256a0381-f439-46cb-a425-1978a16240e0,A Midsummer Night's Sex Comedy,https://en.wikipedia.org/wiki/A_Midsummer_Nigh...,0,It is the early 1900s. Distinguished philosoph...


In [13]:
BATCH_SIZE = 128                                               
                                                                                                           
embeddings = model.encode(                                                                               
    df['text'].tolist(),
    batch_size=BATCH_SIZE,                                                                               
    normalize_embeddings=True,                                                                           
    show_progress_bar=True,                                                                              
    convert_to_numpy=True,                                                                               
)                                                                                                        

df['embedding'] = list(embeddings) 

Batches: 100%|██████████| 172/172 [01:26<00:00,  1.98it/s]


In [14]:
df

,uuid,title,doc_id,chunk_index_in_document,text,embedding
0,2322a7ba-1b14-4bbb-8080-5bb7776667af,Man About Town,https://en.wikipedia.org/wiki/Man_About_Town_(...,0,"Top Hollywood talent agent, Jack Giamoro (Ben ...","[-0.06874588, -0.08719915, -0.009942173, -0.01..."
1,16b41447-6996-436f-9606-91ab3aa8b850,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,0,"Tom Dobbs is host of a satirical news program,...","[-0.06374623, -0.10043635, 0.00068453926, -0.1..."
2,e2fd6cbe-e717-4c69-a6a4-6a989f5aff19,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,1,and given an injection. The next day she displ...,"[-0.08712538, -0.046714358, 0.00083922734, -0...."
3,1a203d4d-ac1f-44ce-a23f-0c4340209621,Man of the Year,https://en.wikipedia.org/wiki/Man_of_the_Year_...,2,feeling that is soon justified when she sees D...,"[-0.027212106, -0.054933857, 0.07684221, -0.06..."
4,446f8747-0a69-41b6-9d6c-3540a759bd83,The Marine,https://en.wikipedia.org/wiki/The_Marine,0,"In Iraq, John Triton (John Cena) of the United...","[-0.12320833, -0.051937085, -0.06665301, -0.08..."
...,...,...,...,...,...,...
21887,956f43a5-4044-470a-aa7b-986bc21ea535,Hammett,https://en.wikipedia.org/wiki/Hammett_(film),0,"San Francisco-based Dashiell Hammett, trying t...","[-0.05523208, 0.025511404, -0.0026574167, 0.00..."
21888,6f6660ae-d54b-4ee4-a4bc-f0387f502326,Hanky Panky,https://en.wikipedia.org/wiki/Hanky_Panky_(198...,0,An architect named Michael Jordon accidentally...,"[-0.009478967, 0.06258904, 0.020682376, -0.000..."
21889,e1e657ca-3f03-4b14-96ce-ce9781767797,Harry Tracy,"https://en.wikipedia.org/wiki/Harry_Tracy,_Des...",0,"By the end of the 19th century, Butch Cassidy,...","[-0.006718236, -0.052259747, -0.058481187, -0...."
21890,256a0381-f439-46cb-a425-1978a16240e0,A Midsummer Night's Sex Comedy,https://en.wikipedia.org/wiki/A_Midsummer_Nigh...,0,It is the early 1900s. Distinguished philosoph...,"[0.0053937477, -0.00034223168, 0.0014458111, 0..."


In [8]:
import random
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient("localhost", port=6333)

# client.delete_collection(collection_name="movies")

client.recreate_collection(
    collection_name="movies",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

/var/folders/wd/k5bk9g_9623_1td2fb7yf1700000gp/T/ipykernel_5781/4035121840.py:9: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [15]:
client.upload_points(
    collection_name="movies",
    points=(
        PointStruct(
            id=row.uuid,
            vector=list(row.embedding),
            payload={"doc_id": row.doc_id, "title": row.title, "chunk_index": row.chunk_index_in_document},
        )
        for row in df.itertuples()
    ),
    batch_size=256,
    parallel=1,
)

In [16]:
query = """A film about a Brazilian jiu-jitsu instructor whose studio window gets shattered when an off-duty 
  ▎ cop's gun accidentally goes off after a lawyer walks in following a car accident.."""
query_vector = model.encode(query).tolist()

result = client.query_points(
    collection_name="movies",
    query=query_vector,
    limit=10,
    with_payload=True,
)

for p in result.points:
    print(round(p.score, 3), p.id, p.payload)

0.545 b93d2c20-5203-40ce-bf3e-963d8513f2d9 {'doc_id': 'https://en.wikipedia.org/wiki/Redbelt', 'title': 'Redbelt', 'chunk_index': 0}
0.462 77b9b46c-0307-4975-b8e9-26a870e90378 {'doc_id': 'https://en.wikipedia.org/wiki/The_Octagon_(film)', 'title': 'The Octagon', 'chunk_index': 0}
0.455 6df8de80-5a4e-4d49-b7c5-eda1dd8c4f92 {'doc_id': 'https://en.wikipedia.org/wiki/Right_at_Your_Door', 'title': 'Right at Your Door', 'chunk_index': 0}
0.439 4a8814d1-8c3f-4c78-93dc-e38899d7facb {'doc_id': 'https://en.wikipedia.org/wiki/They_Still_Call_Me_Bruce', 'title': 'They Still Call Me Bruce', 'chunk_index': 0}
0.435 cc73333e-1eee-46fb-b84d-76ddf94b227d {'doc_id': 'https://en.wikipedia.org/wiki/The_Darwin_Awards_(film)', 'title': 'The Darwin Awards', 'chunk_index': 0}
0.433 aa4f66c7-30bd-4564-ac4e-c9e67fe819cf {'doc_id': 'https://en.wikipedia.org/wiki/McBain_(film)', 'title': 'McBain', 'chunk_index': 0}
0.426 da365dab-a174-4bcf-8cfd-18c1b581f194 {'doc_id': 'https://en.wikipedia.org/wiki/Miracles_(1986